In [179]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [180]:
#Column Cleaning Function
#change column names to be lowercase and replace spaces with underscores
# change $ to usd
def clean_column_names(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('$', 'usd').str.replace('/', 'per').str.replace('&', 'and').str.replace('.', '')
    return df

In [181]:
#Read in test measure data
# 
#df = pd.read_pickle("df_yr1.pkl")

df = pd.read_csv("input/assembled_measures.csv")
clean_column_names(df)
df['incremental_cost'] = df['incremental_installed_cost_(usd)']
df['measure_electric_energy_savings'] = df['energy_impact_1']#df['annual_energy_saved_(kwh)']

#needs to be pulled in to assembled measures
df['kw-kwh_ratio'] = '.75'
df['kw-kwh_ratio'] = df['kw-kwh_ratio'].astype(float)

# we are going to have to report back out the 
#so if energy impacts _units are kwh put them in 'measure_electric_energy_savings'
#  if mmbtu put them in 'measure_natural_gas_savings'
### Need to fix input data to have fuel source
df['measure_natural_gas_savings'] = 100
df['measure_fuel_oil_savings'] = 50
df['measure_propane_savings'] = 50
df['measure_water_savings'] = df['water_savings_(gallons)']


column_names = df.columns.tolist()

print(column_names)

['measure_name', 'sector', 'program', 'market', 'baseline_condition', 'efficient_condition', 'building_type', 'electric_end_use', 'measure_life_(yrs)', 'incremental_installed_cost_(usd)', 'annual_energy_saved_(kwh)', 'water_savings_(gallons)', 'energy_impact_1', 'energy_impact_1_units', 'energy_impact_2', 'energy_impact_2_units', 'energy_impact_3', 'energy_impact_3_units', 'incremental_cost', 'measure_electric_energy_savings', 'kw-kwh_ratio', 'measure_natural_gas_savings', 'measure_fuel_oil_savings', 'measure_propane_savings', 'measure_water_savings']


In [182]:
#read in avoided costs data
avoided_costs = pd.read_excel("input/11_Avoided_Cost.xlsx")
clean_column_names(avoided_costs)
# read in loadshapes data
loadshapes = pd.read_excel("input/12_loadshapes.xlsx")
clean_column_names(loadshapes)
loadshapes
# column_names = avoided_costs.columns.tolist()
# print(column_names)

,condition_name,competition_group,subgroup,summer_on-peak,summer_off-peak,winter_on-peak,winter_off-peak,shoulder_on-peak,shoulder_off-peak,summer_gener_capacity,winter_gener_capacity,summer_tandd,winter_tandd
0,furnace_fuel_oil_existing_residential,heating_cooling,oil_furnace,0.007878,0.018346,0.281812,0.368809,0.129864,0.193292,0.000000,0.381007,0.000000,0.381007
1,furnace_natural_gas_baseline_residential,heating_cooling,gas_furnace,0.007878,0.018346,0.281812,0.368809,0.129864,0.193292,0.000000,0.381007,0.000000,0.381007
2,furnace_natural_gas_efficient_residential,heating_cooling,gas_furnace,0.007878,0.018346,0.281812,0.368809,0.129864,0.193292,0.000000,0.381007,0.000000,0.381007
3,room_ac_electricity_baseline_residential,heating_cooling,room_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
4,room_ac_electricity_efficient_residential,heating_cooling,room_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
5,air_conditioner_electricity_baseline_residential,heating_cooling,central_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
6,air_conditioner_electricity_efficient_residential,heating_cooling,central_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
7,cchp_electricity_efficient_residential,heating_cooling,cchp,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
8,refrigerator_electricity_existing_residential,refrigeration,full_size,0.220863,0.237399,0.095015,0.110339,0.164724,0.171661,1.186104,0.888264,1.186104,0.888264
9,refrigerator_electricity_baseline_residential,refrigeration,full_size,0.220863,0.237399,0.095015,0.110339,0.164724,0.171661,1.186104,0.888264,1.186104,0.888264


In [183]:
def weighted_column_sum(df, weights_row, columns=None, fill_value=0.0):
    """
    df: DataFrame with values to weight (e.g. loadshapes)
    weights_row: Series-like with weights indexed by column name (e.g. avoided_costs.loc[0])
    columns: optional list of columns to use; if None, intersection of df.columns and weights_row.index
    """
    if columns is None:
        columns = df.columns.intersection(weights_row.index)
    w = pd.Series(weights_row).reindex(columns).astype(float).fillna(fill_value)
    return df[columns].fillna(fill_value).dot(w)

#Excel QC complete

In [184]:
### Inputs From Measure Table
# each of these are for a single installation of this measure (one widget)
# all costs and benefits are for just one year
# for now column names created in read in cell above

df['measure_incremental_cost'] = df['incremental_cost']#raw input from measure table

# this is going in the benefits section 

# If measure is in the Ret_ER market than take the cost of the baseline condition and NPV the value by 1/3 of the EUL
# Will need to add all of these values to assembled measures
# inflation rate might be a list 

df["deferred_replacement_credit_savings"] = 0 #df["condition_1_cost"] * (EUL/3) * single_year_inflation_rate
# Needs to be NPV of future avoided replacement costs due to measure extending equipment life


In [185]:
# Line Losses
line_losses = pd.read_excel("input/15_Line_Losses.xlsx")
clean_column_names(line_losses)
line_losses

,sectors,summer_on-peak,summer_off-peak,winter_on-peak,winter_off-peak,shoulder_on-peak,shoulder_off-peak,summer_gener_capacity,winter_gener_capacity,summer_tandd,winter_tandd
0,res,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943
1,com,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790
2,ind,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790


In [186]:
# Electric Energy Savings Calculation with line-loss adjustment
# Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
avoided_costs_electric_use = avoided_costs.iloc[:, :7]
#edit columns to drop some text so we can use the function

avoided_costs_electric_use.columns = [col.replace('_usdperkwh', '') for col in avoided_costs_electric_use.columns]

common = loadshapes.columns.intersection(avoided_costs_electric_use.columns).intersection(line_losses.columns)
# Choose the appropriate row from avoided_costs and line_losses (adjust index/selection if needed)
weights = avoided_costs_electric_use.loc[0, common].astype(float).fillna(0)

losses = line_losses.loc[0, common].astype(float).fillna(0) # in future need to add ability to apply the correct sector

# Adjust weights by (1 - line_loss) so each period is: avoided_cost * (1 - line_loss)
adjusted_weights = weights * (1 - losses)
# Compute the vectorized sum across matching columns: for each row in loadshapes sum(loadshape * adjusted_weight)
period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# Multiply by the measure-level energy savings (broadcasting). If df and loadshapes indices differ
# this will align by index; adjust broadcast strategy if you need a scalar multiplication instead.
df['electric_energy_savings_value'] = period_value * df['measure_electric_energy_savings']
df#excel QC complete

,measure_name,sector,program,market,baseline_condition,efficient_condition,building_type,electric_end_use,measure_life_(yrs),incremental_installed_cost_(usd),...,incremental_cost,measure_electric_energy_savings,kw-kwh_ratio,measure_natural_gas_savings,measure_fuel_oil_savings,measure_propane_savings,measure_water_savings,measure_incremental_cost,deferred_replacement_credit_savings,electric_energy_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,650.0,...,650.0,700.000000,0.75,100,50,50,0.0,650.0,0,29.032807
1,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,650.0,...,650.0,1166.666667,0.75,100,50,50,0.0,650.0,0,48.388011
2,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,650.0,...,650.0,1166.666667,0.75,100,50,50,0.0,650.0,0,48.388011
3,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,650.0,...,650.0,700.000000,0.75,100,50,50,0.0,650.0,0,31.846768
4,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,450.0,450.000000,0.75,100,50,50,0.0,450.0,0,20.472922
5,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,450.0,750.000000,0.75,100,50,50,0.0,450.0,0,34.121537
6,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,450.0,...,450.0,750.000000,0.75,100,50,50,0.0,450.0,0,34.121537
7,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,450.0,...,450.0,450.000000,0.75,100,50,50,0.0,450.0,0,20.472922
8,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,450.0,450.000000,0.75,100,50,50,0.0,450.0,0,18.877415
9,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,450.0,750.000000,0.75,100,50,50,0.0,450.0,0,31.462358


In [187]:
df['electric_demand_savings_value'] = df['kw-kwh_ratio'] * df['measure_electric_energy_savings'] * ( (avoided_costs.loc[0,'summer_gener_capacity_usdperkw-yr'] * (1-line_losses.loc[0,'summer_gener_capacity']) * loadshapes['summer_gener_capacity']) + (avoided_costs.loc[0,'summer_td_usdperkw-yr'] * (1-line_losses.loc[0,'summer_tandd']) * loadshapes['summer_tandd']) + (avoided_costs.loc[0,'winter_gener_capacity_usdperkw-yr'] * (1-line_losses.loc[0,'winter_gener_capacity']) * loadshapes['winter_gener_capacity']) + (avoided_costs.loc[0,'winter_td_usdperkw-yr'] * (1-line_losses.loc[0,'winter_tandd']) * loadshapes['winter_tandd']))
df
# (Energy saved times the kw/kwh ratio (this needs to be added to the measure output) * summer_gen_capacity(CF)from loadshapes * summer_gen_capacity_cost from Avoided cost) same process for winter and T&D sets? then just add together
#line losses all modeled impacts are at the meter all avoided costs are at generation so need to adjust energy or demand impacts from meter to at gen (so this is just multiply)

,measure_name,sector,program,market,baseline_condition,efficient_condition,building_type,electric_end_use,measure_life_(yrs),incremental_installed_cost_(usd),...,measure_electric_energy_savings,kw-kwh_ratio,measure_natural_gas_savings,measure_fuel_oil_savings,measure_propane_savings,measure_water_savings,measure_incremental_cost,deferred_replacement_credit_savings,electric_energy_savings_value,electric_demand_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,650.0,...,700.000000,0.75,100,50,50,0.0,650.0,0,29.032807,0.000000
1,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,650.0,...,1166.666667,0.75,100,50,50,0.0,650.0,0,48.388011,0.000000
2,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,650.0,...,1166.666667,0.75,100,50,50,0.0,650.0,0,48.388011,0.000000
3,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,650.0,...,700.000000,0.75,100,50,50,0.0,650.0,0,31.846768,19199.024932
4,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,450.000000,0.75,100,50,50,0.0,450.0,0,20.472922,12342.230314
5,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,750.000000,0.75,100,50,50,0.0,450.0,0,34.121537,20570.383856
6,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,450.0,...,750.000000,0.75,100,50,50,0.0,450.0,0,34.121537,20570.383856
7,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,450.0,...,450.000000,0.75,100,50,50,0.0,450.0,0,20.472922,12342.230314
8,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,450.000000,0.75,100,50,50,0.0,450.0,0,18.877415,39261.715266
9,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,750.000000,0.75,100,50,50,0.0,450.0,0,31.462358,65436.192110


In [188]:
df['natural_gas_savings_value'] = df['measure_natural_gas_savings'] *avoided_costs.loc[0,'natural_gas_usdpermmbtu']
#raw input from measure table * natural gas avoided cost from avoided costs table 


In [189]:
# df['other_fuel_savings'] = #raw input from measure table * other fuel avoided cost from avoided costs table

#need to make for commercial and industrial as well
df['res_fuel_oil_savings_value'] = df['measure_fuel_oil_savings'] *avoided_costs.loc[0,'res_fuel_oil_usdpermmbtu']
df['res_propane_savings_value'] = df['measure_propane_savings'] *avoided_costs.loc[0,'res_propane_usdpermmbtu']
#desiel and Gasoline?
# df['natural_gas_savings_value'] = df['measure_natural_gas_savings'] *avoided_costs.loc[0,'natural_gas_usdpermmbtu']
# df['natural_gas_savings_value'] = df['measure_natural_gas_savings'] *avoided_costs.loc[0,'natural_gas_usdpermmbtu']

In [190]:
df['other_fuel_savings_value'] = df['res_fuel_oil_savings_value'] + df['res_propane_savings_value'] + df['natural_gas_savings_value']
#  #raw input from measure table * other fuel avoided cost from avoided costs table

In [191]:
# oandm savings are already in the dollar value for a single unit
df['o_and_m_savings_value'] = 0 #df['o_and_m_savings'] 

In [192]:
df['water_savings_value'] = df['measure_water_savings'] *avoided_costs.loc[0,'water_usdpergallon']
#raw input from measure table * water avoided cost from avoided costs table

In [193]:
### Incentives
# Incentive values are a fraction of the measure_incremental_cost depending on program, building type, end_use and install type 
# Input table 13_incentives 
# For now we just do incentives at the measure level and will have the percent tables something in excel other could use
incentives = pd.read_excel("input/13_Incentives.xlsx")
clean_column_names(incentives)
# will need to make sure there is no double counting here
# because are initial counts will be specific to the combination (test_utility_gas & test utility electric) we should
# have a electric and gas utility for each and the incentives need to know this and not add up to more than the incremental cost

#build in a check to make sure these do not add to more than 100 % ? (what about nonutility)
df['electric_utility_incentive'] = df['measure_incremental_cost'] * incentives['electric_incentive_percentage']
df['natural_gas_utility_incentive'] =  df['measure_incremental_cost'] * incentives['natural_gas_incentive_percentage']
df['nonutility_incentive'] = df['measure_incremental_cost'] * incentives['nonutility_incentive_percentage'] 


In [194]:
### Program Costs
# from 14_program costs
programs = pd.read_excel("input/14_Programs.xlsx")
clean_column_names(programs)

# will also need to adjust each utilities program costs allowing for non overlap in the fuel type groupings
# fuel type groups are needed for the tests but they all work together in that all added or all subtracted
# Merge df with programs on 'program' and 'fuel'

#Will need to be edited based on initial stuff from Mike

# for fuel in programs['utility_fuel'].unique():
#     colname = f"{fuel}_utility_nonmeasure_program_cost"
#     # Find the relevant program/fuel cost for each row in df
#     df[colname] = df.apply(
#         lambda row: (
#             row['measure_incremental_cost'] *
#             programs[
#                 (programs['utility_fuel'] == fuel) &
#                 (programs['program'] == row['program'])
#             ]['non-incentive_costs_as_a_percent_of_incentive_costs'].iloc[0]
#             if not programs[
#                 (programs['utility_fuel'] == fuel) &
#                 (programs['program'] == row['program'])
#             ].empty else 0
#         ),
#         axis=1
#     )
# df
df['electric_utility_nonmeasure_program_cost'] = df['measure_incremental_cost'] * programs[(programs['utility_fuel'] == 'electric') & (programs['program'] == 'NLIRRepl')]['non-incentive_costs_as_a_percent_of_incentive_costs'].iloc[0] # from input table 14_program_costs based on program type
df['natural_gas_utility_nonmeasure_program_costs'] = df['measure_incremental_cost'] * programs[programs['utility_fuel'] == 'natural_gas']['non-incentive_costs_as_a_percent_of_incentive_costs'].iloc[0] 
df['nonutility_nonmeasure_program_costs'] = df['measure_incremental_cost'] * programs[programs['utility_fuel'] == 'other']['non-incentive_costs_as_a_percent_of_incentive_costs'].iloc[0] 

In [195]:
## Risk Discount
# going in the benefits section
# Risk Discount Factor to the incremental installed cost, any operation and maintenance costs, 
# and any deferred replacement credit (for early-retirement retrofits)
#0%
#Input from Global inputs sheet in workpapers
# Charactized as a benefit
risk_discount = 0.02
df['risk_discount_value'] = df['measure_incremental_cost'] * risk_discount

""" For each efficiency measure, risk discount costs are calculated by applying the Risk Discount Factor to the incremental installed cost,
any operation and maintenance costs, and any deferred replacement credit (for early-retirement retrofits). 
The societal or total resource cost-effectiveness test costs are added to the societal or total resource benefits ."""

' For each efficiency measure, risk discount costs are calculated by applying the Risk Discount Factor to the incremental installed cost,\nany operation and maintenance costs, and any deferred replacement credit (for early-retirement retrofits). \nThe societal or total resource cost-effectiveness test costs are added to the societal or total resource benefits .'

In [196]:
for ghg, suffix in [('carbon', '_usdpermmbtu_carbon'), ('n2o', '_usdpermmbtu_n2o'), ('ch4', '_usdpermmbtu_ch4')]:
    # Find columns for this gas
    ghg_cols = [col for col in avoided_costs.columns if ghg in col.lower()]
    avoided_costs_ghg = avoided_costs[ghg_cols].copy()
    # Remove suffix from column names
    avoided_costs_ghg.columns = [col.replace(suffix, '') for col in avoided_costs_ghg.columns]
    # Find common columns
    common = loadshapes.columns.intersection(avoided_costs_ghg.columns).intersection(line_losses.columns)
    # Calculate weights and losses
    weights = avoided_costs_ghg.loc[0, common].astype(float).fillna(0)
    losses = line_losses.loc[0, common].astype(float).fillna(0)
    adjusted_weights = weights * (1 - losses)
    period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
    # Save to df
    df[f'electric_{ghg}_savings_value'] = period_value * df['measure_electric_energy_savings']
df
#value should be total value of a single measure 

,measure_name,sector,program,market,baseline_condition,efficient_condition,building_type,electric_end_use,measure_life_(yrs),incremental_installed_cost_(usd),...,electric_utility_incentive,natural_gas_utility_incentive,nonutility_incentive,electric_utility_nonmeasure_program_cost,natural_gas_utility_nonmeasure_program_costs,nonutility_nonmeasure_program_costs,risk_discount_value,electric_carbon_savings_value,electric_n2o_savings_value,electric_ch4_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,650.0,...,325.0,325.0,325.0,195.0,195.0,195.0,13.00,46.361688,0.102195,2.470273
1,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,650.0,...,325.0,325.0,325.0,195.0,195.0,195.0,13.00,77.269480,0.170326,4.117122
2,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,650.0,...,325.0,325.0,325.0,195.0,195.0,195.0,13.00,77.269480,0.170326,4.117122
3,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,650.0,...,325.0,325.0,325.0,195.0,195.0,195.0,13.00,47.526282,0.094513,2.597259
4,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,225.0,225.0,225.0,135.0,135.0,135.0,9.00,30.552610,0.060758,1.669667
5,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,225.0,225.0,225.0,135.0,135.0,135.0,9.00,50.921016,0.101264,2.782778
6,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,450.0,...,225.0,225.0,225.0,135.0,135.0,135.0,9.00,50.921016,0.101264,2.782778
7,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,450.0,...,225.0,225.0,225.0,135.0,135.0,135.0,9.00,30.552610,0.060758,1.669667
8,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,225.0,225.0,225.0,135.0,135.0,135.0,9.00,28.457117,0.057474,1.560437
9,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,225.0,225.0,225.0,135.0,135.0,135.0,9.00,47.428528,0.095789,2.600729


In [197]:
# Use same code but dont multiply by period Value
# pipe losses for natural gas will be addressed in the Excel workbooks

# Need to have input of each fuel type usage
#Will multiply usage by initial cost for each fuel type


# for ghg, suffix in [('carbon', '_usdpermmbtu_carbon'), ('n2o', '_usdpermmbtu_n2o'), ('ch4', '_usdpermmbtu_ch4')]:
#     # Find columns for this gas
#     ghg_cols = [col for col in avoided_costs.columns if ghg in col.lower()]
#     avoided_costs_ghg = avoided_costs[ghg_cols].copy()
#     # Remove suffix from column names
#     avoided_costs_ghg.columns = [col.replace(suffix, '') for col in avoided_costs_ghg.columns]
#     # Find common columns
#     common = loadshapes.columns.intersection(avoided_costs_ghg.columns).intersection(line_losses.columns)
#     # Calculate weights and losses

#     df[f'electric_{ghg}_savings_value'] = period_value * df['measure_electric_energy_savings']
# df


df["end_use_fuel_externalities"] = 0 #df["energy_impact_2"]

In [198]:
# Just a random adder
# In Penn this was DRIPE
#this is the fill in for MeasNonResource Tab 
nonresource_benefits = pd.read_excel("input/16_measnonresource.xlsx")
clean_column_names(nonresource_benefits)
df['other_nonresource_benefits'] = nonresource_benefits.loc[0,'other_nonresource_benefit']
df

,measure_name,sector,program,market,baseline_condition,efficient_condition,building_type,electric_end_use,measure_life_(yrs),incremental_installed_cost_(usd),...,nonutility_incentive,electric_utility_nonmeasure_program_cost,natural_gas_utility_nonmeasure_program_costs,nonutility_nonmeasure_program_costs,risk_discount_value,electric_carbon_savings_value,electric_n2o_savings_value,electric_ch4_savings_value,end_use_fuel_externalities,other_nonresource_benefits
0,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,650.0,...,325.0,195.0,195.0,195.0,13.00,46.361688,0.102195,2.470273,0,0.025395
1,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,650.0,...,325.0,195.0,195.0,195.0,13.00,77.269480,0.170326,4.117122,0,0.025395
2,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,650.0,...,325.0,195.0,195.0,195.0,13.00,77.269480,0.170326,4.117122,0,0.025395
3,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,650.0,...,325.0,195.0,195.0,195.0,13.00,47.526282,0.094513,2.597259,0,0.025395
4,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,225.0,135.0,135.0,135.0,9.00,30.552610,0.060758,1.669667,0,0.025395
5,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,225.0,135.0,135.0,135.0,9.00,50.921016,0.101264,2.782778,0,0.025395
6,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,450.0,...,225.0,135.0,135.0,135.0,9.00,50.921016,0.101264,2.782778,0,0.025395
7,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,450.0,...,225.0,135.0,135.0,135.0,9.00,30.552610,0.060758,1.669667,0,0.025395
8,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,225.0,135.0,135.0,135.0,9.00,28.457117,0.057474,1.560437,0,0.025395
9,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,225.0,135.0,135.0,135.0,9.00,47.428528,0.095789,2.600729,0,0.025395


In [199]:
# I assume the customers for other utiltiies are also used right?
# 
retail_rates = pd.read_excel("input/17_retail_rates.xlsx")
clean_column_names(retail_rates)
# need to have split for sector to change the column it looks at commercial, industrial
df['electric_customer_bill_savings'] = retail_rates.loc[0,'electricity_res_usdperkwh'] * df['measure_electric_energy_savings']
# df['natural_gas_customer_bill_savings'] = retail_rates.loc[0,'natural_gas_res_usdperkwh']
# df['heating_oil_customer_bill_savings'] = retail_rates.loc[0,'heating_oil_res_usdperkwh']
#likely will need the total savings for all fuels but need to check
# 
# #df["customer_bill_savings"]


In [200]:
# All variables All units are in Dollars even savings in real dollars of the starting year

# measure_incremental_cost
# electric_utility_incentive
# electric_utility_nonmeasure_program_cost # need to have these separate as the electric and natural has utilities may be different based on the location of customers
# electric_customer_bill_savings # this is  retail cost of each energy source times energy used
# electric_energy_savings                  # all savings could also be increased useage which would be a negative value instead
# electric_demand_savings                  #- How does T&D and capacity factor in here? (loadshapes?)
# natural_gas_savings
# natural_gas_utility_incentive          # (incentive payment)
# natural_gas_utility_nonmeasure_program_costs
# other_fuel_savings
# nonunitility_incentive
# nonutility_nonmeasure_program_costs
# water_savings
# o_and_m_savings                          #could also be a cost in which case make negitive
# other_nonresource_benefits 
# deferred_replacement_credit_savings   #???????
# risk_discount
# electric_externalities                  # CO2, NOx, CH4
# end_use_fuel_externalities             # Included Natural Gas and Other Fuels



In [201]:
# this is where we consolidate all the df to have all years
#then we split it down to just the first year for the cost tests

This is the equations for the 4 cost tests

In [202]:
# TRC
df["TRC_cost"] = df["measure_incremental_cost"] + df["electric_utility_nonmeasure_program_cost"] + df["natural_gas_utility_nonmeasure_program_costs"] + df["nonutility_incentive"] + df["nonutility_nonmeasure_program_costs"]

df["TRC_benefit"] = df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["other_fuel_savings_value"] + df["water_savings_value"] + df["o_and_m_savings_value"] + df["other_nonresource_benefits"] + df["deferred_replacement_credit_savings"] + df["risk_discount_value"]
df["TRC_BCR"] = df["TRC_benefit"] / df["TRC_cost"]

In [203]:
# SCT
df["SCT_cost"] = df["measure_incremental_cost"] + df["electric_utility_nonmeasure_program_cost"] + df["natural_gas_utility_nonmeasure_program_costs"] + df["nonutility_incentive"] + df["nonutility_nonmeasure_program_costs"]

df["SCT_benefit"] = df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["other_fuel_savings_value"] + df["water_savings_value"] + df["o_and_m_savings_value"] + df["other_nonresource_benefits"] + df["deferred_replacement_credit_savings"] + df["risk_discount_value"] + df["electric_carbon_savings_value"] + df["electric_n2o_savings_value"] + df["electric_ch4_savings_value"] + df["end_use_fuel_externalities"]
df["SCT_BCR"] = df["SCT_benefit"] / df["SCT_cost"]

In [204]:
# RIM
# Why is this just electric bill savings not the other fuels?

 # RIM
df["RIM_cost"] = df["electric_utility_incentive"] + df["electric_utility_nonmeasure_program_cost"] + df["electric_customer_bill_savings"] + df["natural_gas_utility_incentive"] + df["natural_gas_utility_nonmeasure_program_costs"]

df["RIM_benefit"] = df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["other_fuel_savings_value"] + df["nonutility_incentive"] + df["nonutility_nonmeasure_program_costs"]
df["RIM_BCR"] = df["RIM_benefit"] / df["RIM_cost"]

In [205]:
# PCT
# double check the electric customer cost thing I think we need all fuels bill savings or at least an option to change it
df["PCT_cost"] = df["measure_incremental_cost"]

df["PCT_benefit"] = df["electric_utility_incentive"] + df["electric_customer_bill_savings"] + df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["natural_gas_utility_incentive"] + df["other_fuel_savings_value"] + df["nonutility_incentive"] + df["water_savings_value"] + df["o_and_m_savings_value"] + df["other_nonresource_benefits"] + df["deferred_replacement_credit_savings"]
df["PCT_BCR"] = df["PCT_benefit"] / df["PCT_cost"]

In [206]:
df

,measure_name,sector,program,market,baseline_condition,efficient_condition,building_type,electric_end_use,measure_life_(yrs),incremental_installed_cost_(usd),...,TRC_BCR,SCT_cost,SCT_benefit,SCT_BCR,RIM_cost,RIM_benefit,RIM_BCR,PCT_cost,PCT_benefit,PCT_BCR
0,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,650.0,...,2.351960,1560.0,3717.992357,2.383328,1180.000000,4176.032807,3.539011,650.0,4771.058201,7.340090
1,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,650.0,...,2.364368,1560.0,3769.970333,2.416648,1273.333333,4195.388011,3.294807,650.0,4883.746739,7.513457
2,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,650.0,...,2.364368,1560.0,3769.970333,2.416648,1273.333333,4195.388011,3.294807,650.0,4883.746739,7.513457
3,refrigerator_electricity_efficient_residential...,residential,NaN,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,650.0,...,14.660831,1560.0,22921.115149,14.693023,1180.000000,23377.871700,19.811756,650.0,23972.897095,36.881380
4,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,14.813638,1080.0,16031.011666,14.843529,810.000000,16349.703236,20.184819,450.0,16754.728631,37.232730
5,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,22.444936,1080.0,24294.335846,22.494755,870.000000,24591.505393,28.266098,450.0,25056.530788,55.681180
6,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,10.0,450.0,...,22.444936,1080.0,24294.335846,22.494755,870.000000,24591.505393,28.266098,450.0,25056.530788,55.681180
7,refrigerator_electricity_efficient_residential...,residential,NaN,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family,unknown,10.0,450.0,...,14.813638,1080.0,16031.011666,14.843529,810.000000,16349.703236,20.184819,450.0,16754.728631,37.232730
8,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,10.0,450.0,...,39.737609,1080.0,42946.693103,39.765457,810.000000,43267.592681,53.416781,450.0,43672.618076,97.050262
9,refrigerator_electricity_efficient_residential...,residential,NaN,ROB,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,10.0,450.0,...,63.984889,1080.0,69153.804908,64.031301,870.000000,69454.654468,79.832936,450.0,69919.679863,155.377066


In [207]:
df.to_csv("output/measure_costs_benefits.csv", index=False)

This is the output needed for mike to join to the measure database
It is boolen of each cost test pass or fail and than the PCT ratio (key parts)
Will provide everything to Mike

This is an example output needed for deliverable MeasScrn & MeasCostEff

In [208]:
# Measure Name
# Primary Fuel
# 'Program'
# "Measure ID1"
# Sector
# "Building Type/Segment"	
# Primary Fuel 
# End Use	
# "Market(e.g., RET, NC, RENO, REPL)"	
# # Benefits and costs for all 4 tests totaled # done for only the first year of the study
# "Total Resource Benefits" = TRC_benefit
# "Total Resource Costs"  = TRC_cost
# "Total Resource Net Benefits" = TRC_benefit - TRC_cost
# "Total Resource BCR" = TRC_benefit / TRC_cost

In [209]:
# Column names of EMeasure Name
# Primary Fuel
# Include in Calc's
# Measure ID3
# "Measure ID1
# Sector"	
# "Building Type/Segment"	
# Primary Fuel 
# End Use	
# "Market(e.g., RET, NC, RENO, REPL)"
# First Install Year
# Last Install Year
# "Incremental Installed Cost($)"
# "Retrofit deferral credit($)"
# "O&M ($)"
# "Fossil Fuel($)"
# "Fossil Fuel Externalites($)"
# "Risk Mitigation($)"
# "Measure lifetime(years)"
# "Levelized Annual Electric Energy savings(MWh/yr)"
# "Levelized Annual Summer Peak demand savings (kW-yr)"
# "Levelized Annual Winter Peak demand  savings (kW-yr)"
# "Generating Capacity Value of Peak  Demand savings($/kW-yr)"
# "T&D Capacity Value of Peak Demand savings($/kW-yr)"
# "System value of Electric Energy savings($/kWh)"
# "Total Value of Electricity Savings($)"	
# "Environ-mental Externalities($)"	
# "Fossil Fuel($)"	
# "Fossil Fuel Externalities($)"	
# "Water($)"	
# "Total Value of Total Resource Benefits($)"
# "Net Total Resource Benefits($)"
# Total Resource Benefit/ Cost Ratio
# "Net Levelized Cost per kWh ($/kWh)"
# "Net Cost Per Summer Peak kW-yr ($/kW-yr)"
# "Net Cost Per Winter Peak kW-yr ($/kW-yr)"



In [210]:
# #Archive


# carbon_cols = [col for col in avoided_costs.columns if 'carbon' in col.lower()]
# avoided_costs_carbon = avoided_costs[carbon_cols]
# avoided_costs_carbon
# #now edit the carbon avoided columns to match the other tables so we can use the function
# avoided_costs_carbon.columns = [col.replace('_usdpermmbtu_carbon', '') for col in avoided_costs_carbon.columns]
# # Electric Energy Savings Calculation with line-loss adjustment
# # Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
# common = loadshapes.columns.intersection(avoided_costs_carbon.columns).intersection(line_losses.columns)
# # Choose the appropriate row from avoided_costs and line_losses (adjust index/selection if needed)
# weights = avoided_costs_carbon.loc[0, common].astype(float).fillna(0)

# #will need to adjust sector selection based on measure sector
# losses = line_losses.loc[line_losses['sectors'] == 'res', common].iloc[0].astype(float).fillna(0)

# # Adjust weights by (1 - line_loss) so each period is: avoided_cost * (1 - line_loss)
# adjusted_weights = weights * (1 - losses)
# # Compute the vectorized sum across matching columns: for each row in loadshapes sum(loadshape * adjusted_weight)
# period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# # Multiply by the measure-level energy savings (broadcasting). If df and loadshapes indices differ
# # this will align by index; adjust broadcast strategy if you need a scalar multiplication instead.
# df['electric_carbon_savings_value'] = period_value * df['measure_electric_energy_savings']
# df     # CO2, NOx, CH4
# ###########
# n2o_cols = [col for col in avoided_costs.columns if 'n2o' in col.lower()]
# avoided_costs_n2o = avoided_costs[n2o_cols]
# avoided_costs_n2o
# #now edit the carbon avoided columns to match the other tables so we can use the function
# avoided_costs_n2o.columns = [col.replace('_usdpermmbtu_n2o', '') for col in avoided_costs_n2o.columns]
# # Electric Energy Savings Calculation with line-loss adjustment
# # Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
# common = loadshapes.columns.intersection(avoided_costs_n2o.columns).intersection(line_losses.columns)
# # Choose the appropriate row from avoided_costs and line_losses (adjust index/selection if needed)
# weights = avoided_costs_n2o.loc[0, common].astype(float).fillna(0)

# losses = line_losses.loc[line_losses['sectors'] == 'res', common].iloc[0].astype(float).fillna(0)

# # Adjust weights by (1 - line_loss) so each period is: avoided_cost * (1 - line_loss)
# adjusted_weights = weights * (1 - losses)
# # Compute the vectorized sum across matching columns: for each row in loadshapes sum(loadshape * adjusted_weight)
# period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# # Multiply by the measure-level energy savings (broadcasting). If df and loadshapes indices differ
# # this will align by index; adjust broadcast strategy if you need a scalar multiplication instead.
# df['electric_n2o_savings_value'] = period_value * df['measure_electric_energy_savings']
# df     # CO2, NOx, CH4
# ###########
# ch4_cols = [col for col in avoided_costs.columns if '_ch4' in col.lower()]
# avoided_costs_ch4 = avoided_costs[ch4_cols]
# avoided_costs_ch4
# #now edit the carbon avoided columns to match the other tables so we can use the function
# avoided_costs_ch4.columns = [col.replace('_usdpermmbtu_ch4', '') for col in avoided_costs_ch4.columns]
# # Electric Energy Savings Calculation with line-loss adjustment
# # Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
# common = loadshapes.columns.intersection(avoided_costs_ch4.columns).intersection(line_losses.columns)
# # Choose the appropriate row from avoided_costs and line_losses (adjust index/selection if needed)
# weights = avoided_costs_ch4.loc[0, common].astype(float).fillna(0)

# losses = line_losses.loc[line_losses['sectors'] == 'res', common].iloc[0].astype(float).fillna(0)

# # Adjust weights by (1 - line_loss) so each period is: avoided_cost * (1 - line_loss)
# adjusted_weights = weights * (1 - losses)
# # Compute the vectorized sum across matching columns: for each row in loadshapes sum(loadshape * adjusted_weight)
# period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# # Multiply by the measure-level energy savings (broadcasting). If df and loadshapes indices differ
# # this will align by index; adjust broadcast strategy if you need a scalar multiplication instead.
# df['electric_ch4_savings_value'] = period_value * df['measure_electric_energy_savings']
# df     # CO2, NOx, CH4